<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/>
</div>

# EduTrack Analytics
## Notebook 3 — SQL Analytics, KPIs & Performance
### 📝 VERSION APPRENANT

> **Instructions :** Complète les cellules marquées `# TODO`.  
> Les blocs `MÉTHODE` t'expliquent l'approche attendue.  
> Les blocs `🧠 Tes observations` sont à remplir après exécution.

| | |
|---|---|
| **Prérequis** | `inscriptions_analytics.csv` disponible dans `SAVE_PATH` (produit par NB2) |
| **Niveau** | Avancé |
| **Outils** | Python, DuckDB (JupySQL), matplotlib |
| **Durée estimée** | 3h à 4h |

---
> Ce notebook répond aux 4 questions business de la direction EduTrack avec des requêtes SQL sur DuckDB. On produit aussi la heatmap d'activité et l'analyse des revenus par domaine.

---
## Étape 1 — Imports, configuration et connexion DuckDB

> **MÉTHODE — Pourquoi DuckDB plutôt que pandas pour le SQL ?**
>
> DuckDB est un moteur SQL embarqué en Python, sans serveur à installer. Il lit directement les CSV via `read_csv_auto()`. C'est le choix idéal pour les projets analytiques car il permet d'écrire du SQL lisible tout en restant dans l'écosystème Python.
>
> **Avantage sur pandas :** Les requêtes SQL complexes avec `RANK() OVER` ou `LAG() OVER` s'écrivent en une ligne, là où pandas nécessite plusieurs `groupby().transform()`.
>
> **Deux sources de données :**
> - Fichiers bruts (`parcours`, `apprenants`, `sessions`, `paiements`) → depuis GitHub via `BASE_URL`
> - Fichier produit par le NB2 (`inscriptions_analytics.csv`) → depuis `SAVE_PATH` (Drive ou local)

In [ ]:
!pip install jupysql==0.11.1 duckdb-engine --quiet

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import duckdb
import os, sys

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

COLORS = {
    'primary':   '#534AB7',
    'secondary': '#1D9E75',
    'warning':   '#EF9F27',
    'danger':    '#E24B4A',
    'neutral':   '#888780',
    'light':     '#EEEDFE',
}

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#F9F9F8',
    'axes.grid':        True,
    'grid.alpha':       0.35,
    'font.size':        11,
})

# ── Détection Colab / Local ──────────────────────────────────────────────────
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_PATH = '/content/drive/MyDrive/DataProjectLab/projects/elearning_analytics/'
else:
    SAVE_PATH = './outputs/'
os.makedirs(SAVE_PATH, exist_ok=True)
print(f'📁 Environnement : {"Colab" if IN_COLAB else "Local"}')
print(f'📁 Dossier       : {SAVE_PATH}')
print('Configuration chargée ✅')

In [ ]:
BASE_URL   = 'https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/elearning_analytics/data/'
clean_path = f'{SAVE_PATH}inscriptions_analytics.csv'

# Fallback GitHub si le fichier NB2 n'est pas disponible
if not os.path.exists(clean_path):
    clean_path = BASE_URL + 'inscriptions_analytics.csv'
    print('⚠️  inscriptions_analytics.csv non trouvé en local — chargement depuis GitHub')
    print('   → Exécute le Notebook 2 pour produire le fichier nettoyé')
else:
    print(f'✅ Fichier nettoyé trouvé : {clean_path}')

# Connexion DuckDB et chargement des tables
conn = duckdb.connect()
conn.execute(f"""
    CREATE TABLE parcours     AS SELECT * FROM read_csv_auto('{BASE_URL}parcours.csv');
    CREATE TABLE apprenants   AS SELECT * FROM read_csv_auto('{BASE_URL}apprenants.csv',   timestampformat='%Y-%m-%d');
    CREATE TABLE inscriptions AS SELECT * FROM read_csv_auto('{BASE_URL}inscriptions.csv', timestampformat='%Y-%m-%d');
    CREATE TABLE sessions     AS SELECT * FROM read_csv_auto('{BASE_URL}sessions.csv',     timestampformat='%Y-%m-%d');
    CREATE TABLE paiements    AS SELECT * FROM read_csv_auto('{BASE_URL}paiements.csv',    timestampformat='%Y-%m-%d');
    CREATE TABLE inscriptions_analytics AS SELECT * FROM read_csv_auto('{clean_path}', timestampformat='%Y-%m-%d');
""")

result = conn.execute("""
    SELECT 'parcours'                AS table_name, COUNT(*) AS nb FROM parcours     UNION ALL
    SELECT 'apprenants',               COUNT(*) FROM apprenants   UNION ALL
    SELECT 'inscriptions',             COUNT(*) FROM inscriptions UNION ALL
    SELECT 'sessions',                 COUNT(*) FROM sessions     UNION ALL
    SELECT 'paiements',                COUNT(*) FROM paiements    UNION ALL
    SELECT 'inscriptions_analytics',   COUNT(*) FROM inscriptions_analytics
""").df()
display(result)
print('✅ 6 tables chargées')

In [ ]:
# Lier DuckDB à JupySQL pour les cellules %%sql
%load_ext sql
%sql conn --alias duckdb
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
print('%%sql prêt ✅')

---
## Étape 2 — KPIs globaux

### MÉTHODE
Une seule requête pour tous les KPIs globaux — bonne pratique qui garantit la cohérence des chiffres (même filtre, même instant d'exécution). `COUNT(CASE WHEN ... END)` est l'équivalent SQL de `.value_counts()` de pandas, mais plus lisible dans une logique analytique.

In [ ]:
%%sql df_kpi <<
SELECT
    COUNT(DISTINCT a.apprenant_id)   AS total_apprenants,
    COUNT(DISTINCT i.inscription_id) AS total_inscriptions,
    -- TODO : taux_completion_pct (% statut = 'Termine')
    -- TODO : taux_abandon_pct (% statut = 'Abandonne')
    -- TODO : csat_moyen (AVG TRY_CAST sur statut = 'Termine')
    -- TODO : revenu_total_fcfa (SUM paiements valides)
    -- TODO : nb_certificats (SUM certificat_obtenu)
FROM inscriptions i
LEFT JOIN apprenants a   ON i.apprenant_id = a.apprenant_id
LEFT JOIN paiements p    ON i.apprenant_id = p.apprenant_id
                        AND i.parcours_id  = p.parcours_id

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

---
## Étape 3 — Performance par parcours avec RANK()

### MÉTHODE
`RANK() OVER (ORDER BY ...)` est une window function SQL qui attribue un rang à chaque ligne sans réduire le nombre de lignes (contrairement à `GROUP BY`). On l'utilise pour classer les parcours par taux de complétion afin d'identifier les formations les plus efficaces et les plus problématiques.

In [ ]:
%%sql df_perf_parc <<
SELECT
    p.parcours_id,
    p.titre,
    p.domaine,
    p.instructeur,
    p.duree_semaines,
    COUNT(i.inscription_id)   AS nb_inscrits,
    -- TODO : taux_completion (% statut = 'Termine')
    -- TODO : taux_abandon (% statut = 'Abandonne')
    -- TODO : csat_moyen (AVG TRY_CAST sur terminés)
    -- TODO : RANK() OVER (ORDER BY taux_completion DESC) AS rang_completion
FROM parcours p
JOIN inscriptions i ON p.parcours_id = i.parcours_id
GROUP BY p.parcours_id, p.titre, p.domaine, p.instructeur, p.duree_semaines
ORDER BY taux_completion DESC

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
perf_sorted = df_perf_parc.sort_values('taux_completion')

ax.barh(perf_sorted['titre'], perf_sorted['taux_completion'],
        color=COLORS['primary'], alpha=0.85, label='Taux complétion')
ax.barh(perf_sorted['titre'], perf_sorted['taux_abandon'],
        left=perf_sorted['taux_completion'],
        color=COLORS['danger'], alpha=0.6, label='Taux abandon')
ax.axvline(df_perf_parc['taux_completion'].mean(), color=COLORS['warning'],
           linestyle='--', linewidth=1.5,
           label=f"Moyenne complétion {df_perf_parc['taux_completion'].mean():.1f}%")
ax.set_xlabel('Taux (%)')
ax.set_title('Performance des 12 parcours EduTrack', fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(f'{SAVE_PATH}performance_parcours.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Sauvegardé : {SAVE_PATH}performance_parcours.png')

---
## Étape 4 — Performance instructeurs avec RANK()

### MÉTHODE
`RANK() OVER (ORDER BY csat_moyen DESC)` classe les instructeurs par satisfaction. On produit aussi un scatter CSAT × Complétion pour identifier les quadrants de performance.

> **`NTILE(2)` sur deux métriques** : on peut combiner `RANK()` sur le CSAT et sur la complétion pour segmenter les instructeurs en 4 quadrants — Top/Bottom CSAT × Top/Bottom complétion.

In [ ]:
%%sql df_inst <<
SELECT
    p.instructeur,
    COUNT(DISTINCT p.parcours_id)   AS nb_parcours,
    COUNT(i.inscription_id)         AS nb_apprenants,
    -- TODO : csat_moyen (AVG TRY_CAST sur terminés)
    -- TODO : taux_completion
    -- TODO : RANK() OVER (ORDER BY csat_moyen DESC) AS rang_csat
    -- TODO : RANK() OVER (ORDER BY taux_completion DESC) AS rang_completion
FROM parcours p
JOIN inscriptions i ON p.parcours_id = i.parcours_id
GROUP BY p.instructeur
ORDER BY csat_moyen DESC

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(df_inst['taux_completion'], df_inst['csat_moyen'],
           s=df_inst['nb_apprenants']/5, c=COLORS['primary'], alpha=0.7, zorder=5)
med_comp = df_inst['taux_completion'].median()
med_csat = df_inst['csat_moyen'].median()
ax.axvline(med_comp, color=COLORS['neutral'], linestyle='--', linewidth=1, alpha=0.7)
ax.axhline(med_csat, color=COLORS['neutral'], linestyle='--', linewidth=1, alpha=0.7)
for _, row in df_inst.iterrows():
    ax.annotate(row['instructeur'], (row['taux_completion'], row['csat_moyen']),
                textcoords='offset points', xytext=(6, 4), fontsize=9)
ax.text(med_comp + 0.2, med_csat + 0.002, 'Top Performers',
        color=COLORS['secondary'], fontsize=9, fontweight='bold')
ax.set_xlabel('Taux de complétion (%)')
ax.set_ylabel('CSAT moyen')
ax.set_title('Matrice performance instructeurs', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_PATH}matrice_instructeurs.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Sauvegardé : {SAVE_PATH}matrice_instructeurs.png')

---
## Étape 5 — Heatmap activité heure × jour

### MÉTHODE
On extrait le `dayofweek` (0=Lundi à 6=Dimanche) de chaque date de session, puis on pivote pour obtenir une matrice heure × jour. La heatmap révèle les pics d'activité pour optimiser les campagnes relance et la disponibilité du support pédagogique.

> **Pourquoi pandas ici et non DuckDB ?** Le pivot et la visualisation matplotlib nécessitent un DataFrame — on passe par pandas directement plutôt que d'extraire depuis DuckDB puis repivot.

In [ ]:
# Charger sessions en pandas pour la heatmap
import pandas as pd
df_sess = pd.read_csv(BASE_URL + 'sessions.csv', parse_dates=['date_session'])

JOURS = {0: 'Lun', 1: 'Mar', 2: 'Mer', 3: 'Jeu', 4: 'Ven', 5: 'Sam', 6: 'Dim'}
df_sess['jour_semaine'] = df_sess['date_session'].dt.dayofweek

hm = (
    df_sess.groupby(['heure_connexion', 'jour_semaine'])['session_id']
    .count().reset_index()
    .rename(columns={'session_id': 'nb_sessions'})
)
pivot = hm.pivot(index='heure_connexion', columns='jour_semaine', values='nb_sessions').fillna(0)
pivot.columns = [JOURS[c] for c in pivot.columns]

top3 = hm.nlargest(3, 'nb_sessions')
top3['jour_nom'] = top3['jour_semaine'].map(JOURS)
print('Top 3 créneaux d\'activité :')
for _, r in top3.iterrows():
    print(f'  {r["jour_nom"]} {int(r["heure_connexion"])}h : {int(r["nb_sessions"])} sessions')

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))
im = ax.imshow(pivot.values, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(0, len(pivot.index), 2))
ax.set_yticklabels([f'{h}h' for h in pivot.index[::2]])
ax.set_title('Heatmap activité EduTrack — Sessions par heure × jour', fontweight='bold')
ax.set_xlabel('Jour de la semaine')
ax.set_ylabel('Heure de connexion')
plt.colorbar(im, ax=ax, label='Nb sessions')
plt.tight_layout()
plt.savefig(f'{SAVE_PATH}heatmap_activite.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Sauvegardé : {SAVE_PATH}heatmap_activite.png')

---
## Étape 6 — Tendance mensuelle des abandons avec LAG()

### MÉTHODE
`LAG()` est une window function SQL qui retourne la valeur de la ligne précédente dans une partition ordonnée. Ici, `LAG(taux_abandon)` sur la dimension temps permet de calculer la variation mensuelle du taux d'abandon : `variation = taux_actuel - taux_mois_précédent`.

> **DuckDB :** `strftime(date_inscription, '%Y-%m')` extrait l'année-mois depuis une date — équivalent de `dt.to_period('M')` en pandas.

In [ ]:
%%sql df_aband <<
-- TODO : calculer le taux d'abandon mensuel avec LAG()
-- Étape 1 (CTE mensuel) : GROUP BY strftime(date_inscription, '%Y-%m')
--   → compter nb_inscr et nb_aband par mois
-- Étape 2 : calculer taux_abandon = nb_aband / nb_inscr * 100
-- Étape 3 : ajouter variation = taux_actuel - LAG(taux) OVER (ORDER BY mois)
SELECT 'TODO' AS mois, 0 AS nb_inscr, 0 AS nb_aband,
       0.0 AS taux_abandon, NULL AS variation

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
df_aband['mois_dt'] = pd.to_datetime(df_aband['mois'])
ax.plot(df_aband['mois_dt'], df_aband['taux_abandon'],
        color=COLORS['danger'], linewidth=2, marker='o', markersize=4,
        label='Taux abandon mensuel')
ax.axhline(df_aband['taux_abandon'].mean(), color=COLORS['warning'],
           linestyle='--', linewidth=1.5,
           label=f"Moyenne {df_aband['taux_abandon'].mean():.1f}%")
pic = df_aband.nlargest(1, 'taux_abandon')
ax.annotate(f"Pic {pic['taux_abandon'].values[0]}%\n{pic['mois'].values[0]}",
            xy=(pic['mois_dt'].values[0], pic['taux_abandon'].values[0]),
            xytext=(0, 15), textcoords='offset points',
            arrowprops=dict(arrowstyle='->', color=COLORS['danger']),
            color=COLORS['danger'], fontsize=9)
ax.set_xlabel('Mois')
ax.set_ylabel('Taux d\'abandon (%)')
ax.set_title('Tendance mensuelle du taux d\'abandon — EduTrack 2022-2024', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(f'{SAVE_PATH}tendance_abandon.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Sauvegardé : {SAVE_PATH}tendance_abandon.png')

---
## Bilan du Notebook 3

| Question business | Réponse |
|---|---|
| Parcours meilleur complétion | **HTML CSS JavaScript (36.9%)** |
| Parcours taux abandon le plus élevé | **Machine Learning Appliqué (19.6%)** |
| Instructeur le mieux noté CSAT | **Bamba Clarisse / Konaté Aida (4.30/5)** |
| Créneau d'activité le plus élevé | **Dimanche 21h (539 sessions)** |
| Mois pic d'abandon | **Mars 2023 (25.5%)** |

### Patterns SQL maîtrisés

| Pattern | Utilisation |
|---|---|
| `RANK() OVER (ORDER BY)` | Classement parcours et instructeurs |
| `TRY_CAST(csat AS DOUBLE)` | Cast sécurisé sur colonne nullable |
| `COUNT(CASE WHEN ... END)` | KPI conditionnel dans une seule requête |
| `LAG() OVER (ORDER BY mois)` | Variation mensuelle du taux d'abandon |
| `strftime(date, '%Y-%m')` | Extraction mois depuis une date |
| `NULLIF(x, 0)` | Protection division par zéro |

**Pour le NB4 :** Utiliser les fichiers PNG exportés et `inscriptions_analytics.csv` pour construire le dashboard Power BI.

---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.